# Notebook 16b — MJO Three-Way Comparison (lat16, Session 28 architecture)
**Project:** ENSO-BSISO SSL — MJO Extension  
**Author:** Jiayi (jh9141@nyu.edu)

Capstone of the Session 28 lat-aware MJO pipeline. Same three-way comparison as `nb16`, using the **Session 28** embeddings from `nb14b` (sup, constant-3-channel lat prefix + nb14 lon pipeline) and `nb15b` (ssl, same architecture + tightened 25–60 d bandpass). Ablation panels compare both attempts (meridional avg → Session 25 lat-aware → Session 28 lat-aware).

## Three representations

| ID | Object | Source | Dates |
|----|--------|--------|-------|
| `rmm` | Wheeler & Hendon RMM index (conventional) | `rmm_labels.csv` — `atan2(RMM2, RMM1)` | ~16,425 days |
| `sup` | Supervised 2D encoder, lat-aware S28 (nb14b) | `MJO/lat16/results/sup/embeddings.npy` | ~16,425 days |
| `ssl` | SSL temporal 2D encoder, lat-aware S28 (nb15b) | `MJO/lat16/results/ssl/embeddings.npy` | ~16,245 days (after edge drops) |

## Headline questions

1. **Did the Session 28 rewrite resolve the seasonal contamination?** Ablation panel: nb15 (mer. avg) F = 300.84 → nb15b S25 F = 2888.24 → nb15b S28 F = ?
2. **Did phase recovery improve?** nb14 / nb15 / S25 / S28 side-by-side.
3. **Does the SSL ENSO signal survive cleaner preprocessing?** If S28 month F < 50 AND z > 5, the SSL ENSO modulation is genuine intraseasonal coupling.
4. **Did Sup escape the rank-1 collapse?** Check `mean_radius` and PC1 variance share against the Session 27 collapse signatures.

## What this notebook produces

1. **Lag circular correlation** between all 3 pairs (τ ∈ [−30, +30] d), Jammalamadaka & SenGupta ρ_c
2. **Autocorrelation analysis** with e-folding decorrelation timescales
3. **Per-phase ENSO displacement** comparison (3 reps × 8 phases)
4. **Phase composite longitude profiles** (lat-averaged for display) of OLR' for each representation
5. **EN−LN difference composites** (lat-averaged) — the key visual
6. **Lat-resolved EN−LN map** at strongest SSL phase
7. **Three-attempt ablation panel** — nb15 mer. avg vs nb15b S25 vs nb15b S28 on month F and phase val
8. **Auto-generated comparison report** → `MJO/lat16/results/comparison/mjo_comparison_lat16_report.txt`

---

## Cell 1 — Mount Drive + Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

PROJECT_DIR    = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR        = f'{PROJECT_DIR}/MJO'
RAW_DIR        = f'{MJO_DIR}/data/raw'                                    # same raw RMM labels
LAT16_DIR      = f'{MJO_DIR}/lat16'                                        # NEW root
PROCESSED_DIR  = f'{LAT16_DIR}/data/processed'
RESULTS_DIR    = f'{LAT16_DIR}/results'
OUT_DIR        = f'{RESULTS_DIR}/comparison'
os.makedirs(OUT_DIR, exist_ok=True)

# Input files — Session 28 versions
RMM_LABELS_FILE     = f'{RAW_DIR}/rmm_labels.csv'                                # unchanged
MJO_LABELS_FILE     = f'{PROCESSED_DIR}/labels_aligned_mjo_lat16.csv'             # aligned to X_MJO_lat16
SSL_LABELS_FILE     = f'{PROCESSED_DIR}/labels_aligned_mjo_lat16_bp25_60.csv'     # Session 28: bp25-60
X_FILE              = f'{PROCESSED_DIR}/X_MJO_lat16.npy'                          # (N, 3, 16, 180)
LON_FILE            = f'{PROCESSED_DIR}/longitudes_mjo.npy'
LAT_FILE            = f'{PROCESSED_DIR}/latitudes_mjo.npy'
SUP_EMB_FILE        = f'{RESULTS_DIR}/sup/embeddings.npy'
SSL_EMB_FILE        = f'{RESULTS_DIR}/ssl/embeddings.npy'
SUP_PROBE_FILE      = f'{RESULTS_DIR}/sup/linear_probe_results.json'
SSL_PROBE_FILE      = f'{RESULTS_DIR}/ssl/linear_probe_results.json'

print('Input file check:')
for f in [RMM_LABELS_FILE, MJO_LABELS_FILE, SSL_LABELS_FILE,
          X_FILE, LON_FILE, LAT_FILE, SUP_EMB_FILE, SSL_EMB_FILE,
          SUP_PROBE_FILE, SSL_PROBE_FILE]:
    print(f'  {"OK" if os.path.exists(f) else "MISSING":<8s}  {f.replace(PROJECT_DIR, "...")}')
print('\nDrive mounted.')

## Cell 2 — Load All Three Representations

Build `theta_rmm`, `theta_sup`, `theta_ssl` and their date indices. SSL orientation: try both signs, keep the one with positive ρ_c(rmm, ssl; τ=0).

In [ ]:
# --- RMM (conventional) ---
df_rmm = pd.read_csv(RMM_LABELS_FILE, parse_dates=['date'])
df_rmm['date'] = df_rmm['date'].dt.normalize()

# --- Supervised: dates from MJO_LABELS_FILE (aligned to X_MJO_lat16) ---
df_sup = pd.read_csv(MJO_LABELS_FILE, parse_dates=['date'])
df_sup['date'] = df_sup['date'].dt.normalize()
emb_sup = np.load(SUP_EMB_FILE)
assert len(emb_sup) == len(df_sup), f'sup mismatch: {len(emb_sup)} vs {len(df_sup)}'

# --- SSL: dates from SSL_LABELS_FILE (subset after edge drops) ---
df_ssl = pd.read_csv(SSL_LABELS_FILE, parse_dates=['date'])
df_ssl['date'] = df_ssl['date'].dt.normalize()
emb_ssl = np.load(SSL_EMB_FILE)
assert len(emb_ssl) == len(df_ssl), f'ssl mismatch: {len(emb_ssl)} vs {len(df_ssl)}'

# Restrict RMM to the dates we actually have ERA5 for
df_rmm = df_rmm[df_rmm['date'].isin(df_sup['date'])].reset_index(drop=True)
df_rmm = df_rmm.sort_values('date').reset_index(drop=True)
df_sup = df_sup.sort_values('date').reset_index(drop=True)

theta_rmm = np.arctan2(df_rmm['rmm2'].values, df_rmm['rmm1'].values)
theta_sup = np.arctan2(emb_sup[:, 1], emb_sup[:, 0])

# SSL orientation check
def circular_mean(theta):
    return np.arctan2(np.mean(np.sin(theta)), np.mean(np.cos(theta)))

def circular_corr(t1, t2):
    if len(t1) < 5: return np.nan
    s1 = np.sin(t1 - circular_mean(t1))
    s2 = np.sin(t2 - circular_mean(t2))
    den = np.sqrt(np.sum(s1**2) * np.sum(s2**2))
    return float(np.sum(s1 * s2) / den) if den > 0 else 0.0

common_ssl_rmm = df_ssl['date'].isin(df_rmm['date'])
df_ssl_common  = df_ssl[common_ssl_rmm].reset_index(drop=True)
emb_ssl_common = emb_ssl[common_ssl_rmm.values]
date_to_rmm_idx = {d: i for i, d in enumerate(df_rmm['date'].values)}
rmm_idx_for_ssl = np.array([date_to_rmm_idx[d] for d in df_ssl_common['date'].values])

theta_ssl_pos = np.arctan2( emb_ssl_common[:, 1], emb_ssl_common[:, 0])
theta_ssl_neg = np.arctan2(-emb_ssl_common[:, 1], emb_ssl_common[:, 0])
rho_pos = circular_corr(theta_rmm[rmm_idx_for_ssl], theta_ssl_pos)
rho_neg = circular_corr(theta_rmm[rmm_idx_for_ssl], theta_ssl_neg)

if rho_neg > rho_pos:
    print(f'SSL orientation flipped (z₂ negated): ρ_c(rmm,ssl;0) = {rho_neg:.3f}  vs  {rho_pos:.3f} unflipped')
    theta_ssl = np.arctan2(-emb_ssl[:, 1], emb_ssl[:, 0])
    SSL_FLIP = True
else:
    print(f'SSL orientation kept:                  ρ_c(rmm,ssl;0) = {rho_pos:.3f}  vs  {rho_neg:.3f} flipped')
    theta_ssl = np.arctan2( emb_ssl[:, 1], emb_ssl[:, 0])
    SSL_FLIP = False

# Same supervised orientation check (less common, but in case nb14b found a flipped frame)
theta_sup_pos = np.arctan2( emb_sup[:, 1], emb_sup[:, 0])
theta_sup_neg = np.arctan2(-emb_sup[:, 1], emb_sup[:, 0])
rho_sup_pos = circular_corr(theta_rmm, theta_sup_pos)
rho_sup_neg = circular_corr(theta_rmm, theta_sup_neg)
if rho_sup_neg > rho_sup_pos:
    print(f'sup orientation flipped (z₂ negated): ρ_c(rmm,sup;0) = {rho_sup_neg:.3f}  vs  {rho_sup_pos:.3f} unflipped')
    theta_sup = theta_sup_neg
    SUP_FLIP = True
else:
    print(f'sup orientation kept:                  ρ_c(rmm,sup;0) = {rho_sup_pos:.3f}  vs  {rho_sup_neg:.3f} flipped')
    SUP_FLIP = False

dates_rmm = pd.DatetimeIndex(df_rmm['date'].values)
dates_sup = pd.DatetimeIndex(df_sup['date'].values)
dates_ssl = pd.DatetimeIndex(df_ssl['date'].values)

print(f'\nrmm: {len(theta_rmm)} days  {dates_rmm[0].date()} → {dates_rmm[-1].date()}')
print(f'sup: {len(theta_sup)} days  {dates_sup[0].date()} → {dates_sup[-1].date()}')
print(f'ssl: {len(theta_ssl)} days  {dates_ssl[0].date()} → {dates_ssl[-1].date()}')

## Cell 3 — Lag Circular Correlation Helpers

In [ ]:
def precompute_pairs(dates_A, dates_B, max_lag=30):
    """Within-year pair indices (ia, ib) for every lag."""
    date_to_idx_B = {d: i for i, d in enumerate(dates_B)}
    lags = np.arange(-max_lag, max_lag + 1)
    pairs = {}
    for tau in lags:
        ia, ib = [], []
        for i, d in enumerate(dates_A):
            d_shift = d + pd.Timedelta(days=int(tau))
            if d_shift.year != d.year: continue
            if d_shift in date_to_idx_B:
                ia.append(i); ib.append(date_to_idx_B[d_shift])
        pairs[tau] = (np.array(ia, dtype=int), np.array(ib, dtype=int))
    return lags, pairs

def lag_corr_from_pairs(theta_A, theta_B, lags, pairs, min_pairs=30):
    rho     = np.full(len(lags), np.nan)
    n_pairs = np.zeros(len(lags), dtype=int)
    for k, tau in enumerate(lags):
        ia, ib = pairs[tau]
        if len(ia) < min_pairs: continue
        n_pairs[k] = len(ia)
        rho[k] = circular_corr(theta_A[ia], theta_B[ib])
    return rho, n_pairs

def permutation_null_fast(theta_A, theta_B, dates_B, lags, pairs,
                           n_perm=200, quantile=0.95, rng=None):
    if rng is None: rng = np.random.default_rng(42)
    years_B = pd.DatetimeIndex(dates_B).year
    year_groups = {}
    for i, y in enumerate(years_B):
        year_groups.setdefault(y, []).append(i)
    year_groups = {y: np.array(idx) for y, idx in year_groups.items()}

    null_vals = []
    for _ in range(n_perm):
        theta_B_perm = theta_B.copy()
        for idx_arr in year_groups.values():
            theta_B_perm[idx_arr] = theta_B[idx_arr[rng.permutation(len(idx_arr))]]
        rho_perm, _ = lag_corr_from_pairs(theta_A, theta_B_perm, lags, pairs)
        null_vals.extend(np.abs(rho_perm[~np.isnan(rho_perm)]))
    return float(np.quantile(null_vals, quantile))

print('Helpers defined.')

## Cell 4 — Compute Lag Correlations (3 Pairs)

In [ ]:
MAX_LAG = 30
N_PERM  = 200
RNG     = np.random.default_rng(42)

print('Precomputing pair indices...')
t0 = time.time()
lags, pairs_rmm_sup = precompute_pairs(dates_rmm, dates_sup, max_lag=MAX_LAG)
_,    pairs_rmm_ssl = precompute_pairs(dates_rmm, dates_ssl, max_lag=MAX_LAG)
_,    pairs_sup_ssl = precompute_pairs(dates_sup, dates_ssl, max_lag=MAX_LAG)
print(f'  Done in {time.time()-t0:.1f}s')

print('\nComputing lag correlations...')
rho_rmm_sup, n_rmm_sup = lag_corr_from_pairs(theta_rmm, theta_sup, lags, pairs_rmm_sup)
rho_rmm_ssl, n_rmm_ssl = lag_corr_from_pairs(theta_rmm, theta_ssl, lags, pairs_rmm_ssl)
rho_sup_ssl, n_sup_ssl = lag_corr_from_pairs(theta_sup, theta_ssl, lags, pairs_sup_ssl)

for name, rho, n in [('rmm↔sup', rho_rmm_sup, n_rmm_sup),
                      ('rmm↔ssl', rho_rmm_ssl, n_rmm_ssl),
                      ('sup↔ssl', rho_sup_ssl, n_sup_ssl)]:
    peak = np.nanargmax(rho)
    print(f'  {name}: peak ρ={rho[peak]:.3f} at τ={lags[peak]:+d}d  |  '
          f'ρ(τ=0)={rho[lags==0][0]:.3f}  |  N pairs(τ=0)={n[lags==0][0]}')

print(f'\nPermutation null bands ({N_PERM} permutations)...')
t0 = time.time(); null_rmm_sup = permutation_null_fast(theta_rmm, theta_sup, dates_sup, lags, pairs_rmm_sup, n_perm=N_PERM, rng=RNG)
print(f'  rmm↔sup: 95th null |ρ| = {null_rmm_sup:.3f}  ({time.time()-t0:.1f}s)')
t0 = time.time(); null_rmm_ssl = permutation_null_fast(theta_rmm, theta_ssl, dates_ssl, lags, pairs_rmm_ssl, n_perm=N_PERM, rng=RNG)
print(f'  rmm↔ssl: 95th null |ρ| = {null_rmm_ssl:.3f}  ({time.time()-t0:.1f}s)')
t0 = time.time(); null_sup_ssl = permutation_null_fast(theta_sup, theta_ssl, dates_ssl, lags, pairs_sup_ssl, n_perm=N_PERM, rng=RNG)
print(f'  sup↔ssl: 95th null |ρ| = {null_sup_ssl:.3f}  ({time.time()-t0:.1f}s)')

## Cell 5 — Plot 3-Panel Lag Correlation + Overlay

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('MJO lat16 — Pairwise Lag Circular Correlation ρ_c(A, B; τ)\n'
             'τ > 0: A leads B  |  within-year pairs  |  θ = atan2(z₂, z₁)'
             + (' (SSL z₂ negated)' if SSL_FLIP else '')
             + (' (sup z₂ negated)' if SUP_FLIP else ''),
             fontsize=12, fontweight='bold')

panels = [
    (rho_rmm_sup, null_rmm_sup, n_rmm_sup, 'ρ_c(RMM index,  Supervised lat16)', '#1f77b4'),
    (rho_rmm_ssl, null_rmm_ssl, n_rmm_ssl, 'ρ_c(RMM index,  SSL lat16)',         '#2ca02c'),
    (rho_sup_ssl, null_sup_ssl, n_sup_ssl, 'ρ_c(Supervised lat16,  SSL lat16)',  '#d62728'),
]

for ax, (rho, nb, npairs, title, color) in zip(axes, panels):
    ax.plot(lags, rho, color=color, lw=2, zorder=3)
    ax.fill_between(lags, rho, 0, where=np.abs(rho) > nb, color=color, alpha=0.25, zorder=2)
    ax.axhline( nb, color='gray', ls='--', lw=1, alpha=0.7, label=f'95% null ({nb:.3f})')
    ax.axhline(-nb, color='gray', ls='--', lw=1, alpha=0.7)
    ax.axhline(0,   color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8, ls=':')
    peak = np.nanargmax(rho)
    ax.annotate(f'peak τ={lags[peak]:+d}d\nρ={rho[peak]:.3f}',
                xy=(lags[peak], rho[peak]),
                xytext=(lags[peak] + 5*np.sign(lags[peak] - 1), rho[peak] - 0.06),
                fontsize=9, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2))
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Lag τ (days)'); ax.set_ylabel('ρ_c')
    ax.set_xlim(-MAX_LAG, MAX_LAG); ax.set_ylim(-0.5, 1.0)
    ax.legend(fontsize=8, loc='lower right'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/lag_circular_corr.png', dpi=150, bbox_inches='tight')
plt.show()

# Overlay
fig, ax = plt.subplots(figsize=(10, 5))
for rho, nb, label, color in [
    (rho_rmm_sup, null_rmm_sup, 'rmm ↔ sup-lat16', '#1f77b4'),
    (rho_rmm_ssl, null_rmm_ssl, 'rmm ↔ ssl-lat16', '#2ca02c'),
    (rho_sup_ssl, null_sup_ssl, 'sup-lat16 ↔ ssl-lat16', '#d62728'),
]:
    ax.plot(lags, rho, color=color, lw=2, label=label)
    ax.axhline(nb, color=color, lw=0.8, ls=':', alpha=0.6)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Lag τ (days)  [τ > 0: A leads B]'); ax.set_ylabel('ρ_c')
ax.set_title('MJO Lat16 Three-Way Lag Circular Correlation', fontsize=12)
ax.set_xlim(-MAX_LAG, MAX_LAG); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/lag_circular_corr_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: lag_circular_corr.png, lag_circular_corr_overlay.png')

## Cell 5b — Autocorrelation Analysis (Temporal Self-Memory)

Same diagnostic as nb16 cell 5b. e-folding decorrelation timescale τ_e = first positive lag where ρ_auto drops to e⁻¹ ≈ 0.368. Observed MJO coherence: τ_e ≈ 15–25 d.

In [ ]:
print('Precomputing autocorrelation pair indices...')
t0 = time.time()
_, auto_pairs_rmm = precompute_pairs(dates_rmm, dates_rmm, max_lag=MAX_LAG)
_, auto_pairs_sup = precompute_pairs(dates_sup, dates_sup, max_lag=MAX_LAG)
_, auto_pairs_ssl = precompute_pairs(dates_ssl, dates_ssl, max_lag=MAX_LAG)
print(f'  Done in {time.time()-t0:.1f}s')

rho_auto_rmm, n_auto_rmm = lag_corr_from_pairs(theta_rmm, theta_rmm, lags, auto_pairs_rmm)
rho_auto_sup, n_auto_sup = lag_corr_from_pairs(theta_sup, theta_sup, lags, auto_pairs_sup)
rho_auto_ssl, n_auto_ssl = lag_corr_from_pairs(theta_ssl, theta_ssl, lags, auto_pairs_ssl)

print('Permutation nulls for autocorrelations...')
t0 = time.time()
null_auto_rmm = permutation_null_fast(theta_rmm, theta_rmm, dates_rmm, lags, auto_pairs_rmm, n_perm=N_PERM, rng=RNG)
null_auto_sup = permutation_null_fast(theta_sup, theta_sup, dates_sup, lags, auto_pairs_sup, n_perm=N_PERM, rng=RNG)
null_auto_ssl = permutation_null_fast(theta_ssl, theta_ssl, dates_ssl, lags, auto_pairs_ssl, n_perm=N_PERM, rng=RNG)
print(f'  rmm: 95th null |ρ| = {null_auto_rmm:.3f}  ({time.time()-t0:.1f}s)')
print(f'  sup: 95th null |ρ| = {null_auto_sup:.3f}')
print(f'  ssl: 95th null |ρ| = {null_auto_ssl:.3f}')

def decorrelation_time(rho, lags, threshold=1.0/np.e):
    for tau, r in zip(lags[lags > 0], rho[lags > 0]):
        if not np.isnan(r) and r <= threshold:
            return int(tau)
    return int(lags[-1]) + 1

tau_e_rmm = decorrelation_time(rho_auto_rmm, lags)
tau_e_sup = decorrelation_time(rho_auto_sup, lags)
tau_e_ssl = decorrelation_time(rho_auto_ssl, lags)

print(f'\nAutocorrelation summary  (τ > 0):')
print(f'{"Rep":<8s}  {"ρ(τ=1)":>8s}  {"ρ(τ=5)":>8s}  {"ρ(τ=10)":>8s}  {"ρ(τ=20)":>8s}  {"τ_e":>8s}')
for rep, rho, tau_e in [('rmm', rho_auto_rmm, tau_e_rmm),
                         ('sup', rho_auto_sup, tau_e_sup),
                         ('ssl', rho_auto_ssl, tau_e_ssl)]:
    def g(t):
        idx = np.where(lags == t)[0]
        return f'{rho[idx[0]]:.3f}' if len(idx) and not np.isnan(rho[idx[0]]) else '  n/a'
    print(f'{rep:<8s}  {g(1):>8s}  {g(5):>8s}  {g(10):>8s}  {g(20):>8s}  {str(tau_e)+"d":>8s}')

# Plot
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('MJO Lat16 — Circular Autocorrelation ρ_c(A(t), A(t+τ))\n'
             'τ ≥ 0 only  |  within-year pairs',
             fontsize=12, fontweight='bold')

e_fold   = 1.0 / np.e
pos_mask = lags >= 0
lags_pos = lags[pos_mask]

for rho, nb, label, color in [
    (rho_auto_rmm, null_auto_rmm, f'RMM  (τ_e = {tau_e_rmm}d)',         '#1f77b4'),
    (rho_auto_sup, null_auto_sup, f'Sup-lat16  (τ_e = {tau_e_sup}d)',   '#d62728'),
    (rho_auto_ssl, null_auto_ssl, f'SSL-lat16  (τ_e = {tau_e_ssl}d)',   '#2ca02c'),
]:
    ax_left.plot(lags_pos, rho[pos_mask], color=color, lw=2, label=label)
    ax_left.fill_between(lags_pos, rho[pos_mask], e_fold,
                         where=rho[pos_mask] >= e_fold,
                         color=color, alpha=0.15, interpolate=True)
    ax_left.axhline(nb, color=color, lw=0.8, ls=':', alpha=0.6)

ax_left.axhline(e_fold, color='gray', ls='--', lw=1.2, label=f'e⁻¹ ≈ {e_fold:.2f}')
ax_left.axhline(0, color='k', lw=0.6)
ax_left.set_xlabel('Lag τ (days)'); ax_left.set_ylabel('ρ_c')
ax_left.set_title('Autocorrelation Overlay', fontsize=11)
ax_left.set_xlim(0, MAX_LAG); ax_left.set_ylim(-0.15, 1.05)
ax_left.legend(fontsize=9); ax_left.grid(True, alpha=0.3)

tau_vals   = [tau_e_rmm, tau_e_sup, tau_e_ssl]
bar_labels = ['RMM\n(conventional)', 'Sup-lat16\n(nb14b)', 'SSL-lat16\n(nb15b)']
bar_colors = ['#1f77b4', '#d62728', '#2ca02c']
bars = ax_right.bar(bar_labels, tau_vals, color=bar_colors, alpha=0.85, width=0.5)
for bar, v in zip(bars, tau_vals):
    ax_right.text(bar.get_x() + bar.get_width() / 2, v + 0.3,
                  f'{v}d', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax_right.set_ylabel('e-folding time τ_e (days)')
ax_right.set_title('Temporal Memory: e-Folding Decorrelation Time', fontsize=11)
ax_right.set_ylim(0, max(tau_vals) * 1.3 + 2)
ax_right.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/autocorrelation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: autocorrelation.png')

## Cell 6 — Per-Phase ENSO Displacement Z-Scores (3 Reps × 8 Phases)

Same procedure as nb16. RMM uses discrete `phase` column; sup and ssl sectored from atan2 angles.

In [ ]:
def angle_to_sector(theta):
    deg = np.degrees(theta) + 180  # → [0, 360)
    return ((deg // 45).astype(int) % 8) + 1

def compute_displacement_zscore(emb, sectors, enso, n_perm=1000, rng=None):
    if rng is None: rng = np.random.default_rng(42)
    sectors = np.asarray(sectors); enso = np.asarray(enso)
    disp = np.full(8, np.nan)
    for s in range(1, 9):
        mEN = (sectors == s) & (enso == 'El Nino')
        mLN = (sectors == s) & (enso == 'La Nina')
        if mEN.sum() < 3 or mLN.sum() < 3: continue
        disp[s-1] = np.linalg.norm(emb[mEN].mean(axis=0) - emb[mLN].mean(axis=0))
    obs_mu = float(np.nanmean(disp))

    null = []
    for _ in range(n_perm):
        shuf = enso[rng.permutation(len(enso))]
        d = []
        for s in range(1, 9):
            mEN = (sectors == s) & (shuf == 'El Nino')
            mLN = (sectors == s) & (shuf == 'La Nina')
            if mEN.sum() < 3 or mLN.sum() < 3: continue
            d.append(np.linalg.norm(emb[mEN].mean(axis=0) - emb[mLN].mean(axis=0)))
        if d: null.append(np.mean(d))
    bmu = float(np.mean(null)); bsd = float(np.std(null))
    z = float((obs_mu - bmu) / (bsd + 1e-8))
    return disp, obs_mu, bmu, bsd, z

# RMM
emb_rmm = np.stack([df_rmm['rmm1'].values, df_rmm['rmm2'].values], axis=1)
act_rmm = (~df_rmm['weak_mjo'].values) & (df_rmm['phase'].between(1, 8).values)
disp_rmm, mu_rmm, b_rmm, s_rmm, z_rmm = compute_displacement_zscore(
    emb_rmm[act_rmm], df_rmm.loc[act_rmm, 'phase'].values,
    df_rmm.loc[act_rmm, 'enso_category'].values, n_perm=1000)

# Supervised lat16
emb_sup_use = emb_sup.copy()
if SUP_FLIP: emb_sup_use[:, 1] = -emb_sup_use[:, 1]
sec_sup = angle_to_sector(theta_sup)
act_sup = (~df_sup['weak_mjo'].values) & (df_sup['phase'].between(1, 8).values)
disp_sup, mu_sup, b_sup, s_sup, z_sup = compute_displacement_zscore(
    emb_sup_use[act_sup], sec_sup[act_sup],
    df_sup.loc[act_sup, 'enso_category'].values, n_perm=1000)

# SSL lat16
emb_ssl_use = emb_ssl.copy()
if SSL_FLIP: emb_ssl_use[:, 1] = -emb_ssl_use[:, 1]
sec_ssl = angle_to_sector(theta_ssl)
act_ssl = (~df_ssl['weak_mjo'].values) & (df_ssl['phase'].between(1, 8).values)
disp_ssl, mu_ssl, b_ssl, s_ssl, z_ssl = compute_displacement_zscore(
    emb_ssl_use[act_ssl], sec_ssl[act_ssl],
    df_ssl.loc[act_ssl, 'enso_category'].values, n_perm=1000)

print('=' * 80)
print('PER-PHASE ENSO DISPLACEMENT (active MJO only, 1000 permutations)')
print('=' * 80)
print(f'{"Rep":<14s} {"obs_mu":>10s} {"null_mu":>10s} {"null_sd":>10s} {"z-score":>10s}')
for rep, (mu, b, s, z) in [('rmm',         (mu_rmm, b_rmm, s_rmm, z_rmm)),
                            ('sup (lat16)', (mu_sup, b_sup, s_sup, z_sup)),
                            ('ssl (lat16)', (mu_ssl, b_ssl, s_ssl, z_ssl))]:
    print(f'{rep:<14s} {mu:>10.4f} {b:>10.4f} {s:>10.4f} {z:>10.2f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
for ax, (rep, disp, b, s, z) in zip(axes, [
    ('RMM',             disp_rmm, b_rmm, s_rmm, z_rmm),
    ('Sup-lat16 (nb14b)', disp_sup, b_sup, s_sup, z_sup),
    ('SSL-lat16 (nb15b)', disp_ssl, b_ssl, s_ssl, z_ssl)]):
    phases = np.arange(1, 9)
    valid = ~np.isnan(disp)
    ax.bar(phases[valid], disp[valid], color='steelblue', alpha=0.85)
    ax.axhline(b, color='red', ls='--', lw=1.5, label=f'Null mean ({b:.3f})')
    ax.axhline(b + 2*s, color='red', ls=':', lw=1, label='Null +2σ')
    ax.set_xticks(phases); ax.set_xlabel('Phase / Sector')
    ax.set_ylabel('||EN−LN||')
    ax.set_title(f'{rep}:  z = {z:.2f}', fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('MJO Lat16 — ENSO Displacement per Phase  (3 representations)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/enso_displacement_3way.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: enso_displacement_3way.png')

## Cell 7 — Phase Composite Longitude Profiles (OLR' per representation)

For each rep × phase, mean OLR' anomaly longitude profile. With lat-aware data the underlying composites are 2D `(lat, lon)`, but for direct comparison with nb16's 1D profiles we **meridionally average for display only**. The saved tensor itself retains the full (lat, lon) field.

In [ ]:
X_full = np.load(X_FILE)   # (N_full, 3, 16, 180), channels [u850, OLR, u200]
lons   = np.load(LON_FILE)
lats   = np.load(LAT_FILE)
print(f'X_full shape: {X_full.shape}  (expect (N, 3, 16, 180))')

# X_full aligned to df_sup
date_to_xrow = {d: i for i, d in enumerate(df_sup['date'].values)}

rep_specs = [
    ('RMM',              df_rmm['phase'].values, df_rmm['date'].values, df_rmm['weak_mjo'].values),
    ('Sup-lat16 (nb14b)', sec_sup,                df_sup['date'].values, df_sup['weak_mjo'].values),
    ('SSL-lat16 (nb15b)', sec_ssl,                df_ssl['date'].values, df_ssl['weak_mjo'].values),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig.suptitle("MJO Lat16 — Phase Composite OLR' Longitude Profiles  (meridionally averaged for display)\n"
             "rows = representations, lines = phases",
             fontsize=13, fontweight='bold')

phase_colors = plt.cm.viridis(np.linspace(0, 0.95, 8))

for ax, (rep_name, sectors, dates_, weak) in zip(axes, rep_specs):
    for s in range(1, 9):
        active_mask = (sectors == s) & (~weak)
        sel_dates = dates_[active_mask]
        rows = [date_to_xrow[d] for d in sel_dates if d in date_to_xrow]
        if len(rows) == 0: continue
        # channel 1 = OLR; mean over (samples, lat)
        olr_comp = X_full[rows, 1].mean(axis=(0, 1))   # (180,)
        ax.plot(lons, olr_comp, color=phase_colors[s-1], lw=1.5, label=f'P{s} (n={len(rows)})')
    ax.axhline(0, color='k', lw=0.5, alpha=0.4)
    ax.set_ylabel(f'{rep_name}\nOLR′ (σ)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, ncol=4, loc='upper right')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Longitude (°)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/phase_composites_olr.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: phase_composites_olr.png')

## Cell 8 — EN−LN Difference Composites

For each rep × phase: composite OLR' for EN days minus LN days. Meridionally averaged for display to keep 8×180 panels (same format as nb16).

In [ ]:
def get_enln_diff_profile(sectors, dates_, enso, weak, channel=1):
    """Returns (8, n_lon) array of EN-LN OLR' difference per phase.
    Lat axis is averaged for display only."""
    diff = np.full((8, len(lons)), np.nan)
    for s in range(1, 9):
        base = (sectors == s) & (~weak)
        d_en = dates_[base & (enso == 'El Nino')]
        d_ln = dates_[base & (enso == 'La Nina')]
        rows_en = [date_to_xrow[d] for d in d_en if d in date_to_xrow]
        rows_ln = [date_to_xrow[d] for d in d_ln if d in date_to_xrow]
        if len(rows_en) < 3 or len(rows_ln) < 3: continue
        # Mean over (samples, lat) → 1D longitude profile
        diff[s-1] = (X_full[rows_en, channel].mean(axis=(0, 1))
                     - X_full[rows_ln, channel].mean(axis=(0, 1)))
    return diff

diff_rmm = get_enln_diff_profile(df_rmm['phase'].values, df_rmm['date'].values,
                                   df_rmm['enso_category'].values, df_rmm['weak_mjo'].values)
diff_sup = get_enln_diff_profile(sec_sup, df_sup['date'].values,
                                   df_sup['enso_category'].values, df_sup['weak_mjo'].values)
diff_ssl = get_enln_diff_profile(sec_ssl, df_ssl['date'].values,
                                   df_ssl['enso_category'].values, df_ssl['weak_mjo'].values)

vmax = max(np.nanmax(np.abs(d)) for d in [diff_rmm, diff_sup, diff_ssl])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("MJO Lat16 — EN − LN  OLR' Difference  (per phase, per representation)\n"
             'Meridionally averaged for display.  Larger / more coherent patches → stronger ENSO modulation of MJO',
             fontsize=12, fontweight='bold')

for ax, (rep, d, z) in zip(axes,
                            [('RMM',             diff_rmm, z_rmm),
                             ('Sup-lat16',       diff_sup, z_sup),
                             ('SSL-lat16',       diff_ssl, z_ssl)]):
    im = ax.imshow(d, cmap='RdBu_r', aspect='auto',
                   extent=[lons.min(), lons.max(), 8.5, 0.5],
                   vmin=-vmax, vmax=vmax)
    ax.set_yticks(range(1, 9))
    ax.set_xlabel('Longitude (°)')
    ax.set_ylabel('Phase / Sector')
    ax.set_title(f'{rep}  (rep-level z = {z:.2f})', fontweight='bold')
    plt.colorbar(im, ax=ax, label='EN − LN (σ)', fraction=0.04, pad=0.04)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/enln_difference_composites.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: enln_difference_composites.png')

print('\nMax |EN−LN| in OLR composites (σ units, lat-averaged):')
for rep, d in [('RMM', diff_rmm), ('Sup-lat16', diff_sup), ('SSL-lat16', diff_ssl)]:
    print(f'  {rep:<11s}: {np.nanmax(np.abs(d)):.3f}σ')

## Cell 8b — Lat-Resolved EN−LN Composite for the Strongest Phase  *(new with lat16)*

Diagnostic that **was not possible in nb16**: pick the phase with the largest |EN−LN| amplitude for SSL, then show the full lat×lon EN−LN map (not lat-averaged). Lets us see where the lat-aware encoder is finding signal — equatorial Kelvin band, off-equator Rossby gyres, ITCZ asymmetry, or something else.

In [ ]:
def get_enln_diff_map(sectors, dates_, enso, weak, channel=1):
    """Returns (8, n_lat, n_lon) EN-LN OLR' difference per phase, lat-resolved."""
    diff = np.full((8, X_full.shape[2], X_full.shape[3]), np.nan, dtype=np.float32)
    for s in range(1, 9):
        base = (sectors == s) & (~weak)
        d_en = dates_[base & (enso == 'El Nino')]
        d_ln = dates_[base & (enso == 'La Nina')]
        rows_en = [date_to_xrow[d] for d in d_en if d in date_to_xrow]
        rows_ln = [date_to_xrow[d] for d in d_ln if d in date_to_xrow]
        if len(rows_en) < 3 or len(rows_ln) < 3: continue
        diff[s-1] = (X_full[rows_en, channel].mean(axis=0)
                     - X_full[rows_ln, channel].mean(axis=0))
    return diff

diff_rmm_2d = get_enln_diff_map(df_rmm['phase'].values, df_rmm['date'].values,
                                  df_rmm['enso_category'].values, df_rmm['weak_mjo'].values)
diff_sup_2d = get_enln_diff_map(sec_sup, df_sup['date'].values,
                                  df_sup['enso_category'].values, df_sup['weak_mjo'].values)
diff_ssl_2d = get_enln_diff_map(sec_ssl, df_ssl['date'].values,
                                  df_ssl['enso_category'].values, df_ssl['weak_mjo'].values)

# Pick the strongest phase for SSL (where the EN-LN signal is largest)
ssl_phase_strengths = np.array([np.nanmax(np.abs(diff_ssl_2d[p])) if not np.all(np.isnan(diff_ssl_2d[p])) else 0.0
                                 for p in range(8)])
ph_star = int(np.argmax(ssl_phase_strengths)) + 1
print(f'Strongest SSL phase by max |EN-LN|: P{ph_star} (|Δ|_max = {ssl_phase_strengths[ph_star-1]:.3f}σ)')
print(f'\nPer-phase max |EN-LN| (σ, lat-resolved):')
print(f'{"Phase":>6s} {"RMM":>8s} {"Sup":>8s} {"SSL":>8s}')
for p in range(8):
    r_v = np.nanmax(np.abs(diff_rmm_2d[p])) if not np.all(np.isnan(diff_rmm_2d[p])) else np.nan
    s_v = np.nanmax(np.abs(diff_sup_2d[p])) if not np.all(np.isnan(diff_sup_2d[p])) else np.nan
    l_v = ssl_phase_strengths[p]
    print(f'{p+1:>6d} {r_v:>8.3f} {s_v:>8.3f} {l_v:>8.3f}')

# Plot lat-resolved EN-LN maps for ph_star, all three reps side-by-side
vmax2 = max(np.nanmax(np.abs(d[ph_star-1])) for d in [diff_rmm_2d, diff_sup_2d, diff_ssl_2d])
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True, sharey=True)
fig.suptitle(f"MJO Lat16 — Lat-Resolved EN − LN OLR' map at SSL's strongest phase P{ph_star}\n"
             '(diagnostic only available because nb13b preserved the lat axis)',
             fontsize=12, fontweight='bold')

for ax, (rep_name, diff2d, z) in zip(axes,
                                       [('RMM',       diff_rmm_2d, z_rmm),
                                        ('Sup-lat16', diff_sup_2d, z_sup),
                                        ('SSL-lat16', diff_ssl_2d, z_ssl)]):
    panel = diff2d[ph_star-1]
    im = ax.imshow(panel, cmap='RdBu_r', aspect='auto',
                   extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                   vmin=-vmax2, vmax=vmax2, origin='lower')
    ax.set_title(f'{rep_name}  (rep z = {z:.2f}, max|Δ| = {np.nanmax(np.abs(panel)):.3f}σ)',
                 fontsize=11, fontweight='bold')
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
    ax.set_ylabel('Latitude (°)')

axes[-1].set_xlabel('Longitude (°)')
fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.94, 0.15, 0.015, 0.7])
plt.colorbar(im, cax=cbar_ax, label="EN − LN OLR' (σ)")
fig_path = f'{OUT_DIR}/enln_lat_resolved_strongest_phase.png'
plt.savefig(fig_path, dpi=140, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## Cell 9 — Ablation Panels: nb15 vs nb15b (month F) + nb14 vs nb14b (phase val)

**Session 25's two headline targets.** We compare the meridionally-averaged baselines (nb14, nb15) against the lat-aware redesigns (nb14b, nb15b).

- **Left panel**: SSL angle ANOVA F by month. Did the lat-aware redesign cut nb15's F = 300.84 below the 50 threshold?
- **Right panel**: Supervised and SSL phase val. Did both rise (target nb14b > 60%, nb15b > 30%)?

Baseline values (nb14, nb15) come from the conversation log; lat16 values are computed live in this notebook.

In [ ]:
from scipy.stats import f_oneway

# Compute live S28 month F for SSL angle (this notebook's nb15b run)
months_ssl = pd.DatetimeIndex(df_ssl['date'].values).month
angles_ssl_by_month = [theta_ssl[months_ssl == m] for m in range(1, 13)]
f_ssl_S28, p_ssl_S28 = f_oneway(*angles_ssl_by_month)

# Sup angle by month (informational, S28 only)
months_sup = pd.DatetimeIndex(df_sup['date'].values).month
angles_sup_by_month = [theta_sup[months_sup == m] for m in range(1, 13)]
f_sup_S28, p_sup_S28 = f_oneway(*angles_sup_by_month)

# Load S28 phase val from JSON files (this run)
with open(SUP_PROBE_FILE) as f:
    sup_probe = json.load(f)
with open(SSL_PROBE_FILE) as f:
    ssl_probe = json.load(f)
sup_phase_S28 = sup_probe['RMM Phase']['val_acc']
ssl_phase_S28 = ssl_probe['RMM Phase']['val_acc']

# Baselines from the conversation log
NB14_PHASE       = 0.577     # nb14 sup phase val (meridional avg)
NB15_PHASE       = 0.247     # nb15 ssl phase val (meridional avg)
NB15_MONTH_F     = 300.84    # nb15 ssl angle ANOVA F (meridional avg)
# Session 25 attempt (failed; documented in Session 27 / Desktop/DDCS run)
NB14B_S25_PHASE  = 0.363     # Session 25 sup phase val
NB15B_S25_PHASE  = 0.182     # Session 25 ssl phase val
NB15B_S25_MONTH_F = 2888.24  # Session 25 ssl angle ANOVA F

print('=' * 70)
print('ABLATION: meridional avg (nb14/nb15) -> Session 25 lat-aware (failed) -> Session 28 lat-aware (this run)')
print('=' * 70)
print(f'SSL angle ANOVA F by month:')
print(f'  nb15 (mer. avg)        : {NB15_MONTH_F:7.2f}')
print(f'  nb15b Session 25       : {NB15B_S25_MONTH_F:7.2f}  (worse by {NB15B_S25_MONTH_F - NB15_MONTH_F:+.0f})')
print(f'  nb15b Session 28 (this): {f_ssl_S28:7.2f}  (delta from S25: {f_ssl_S28 - NB15B_S25_MONTH_F:+.2f})')
print(f'\nSup angle ANOVA F by month (S28 only, informational):  {f_sup_S28:.2f}')
print(f'\nSSL phase val:')
print(f'  nb15 (mer. avg)        : {NB15_PHASE*100:5.1f}%')
print(f'  nb15b Session 25       : {NB15B_S25_PHASE*100:5.1f}%')
print(f'  nb15b Session 28 (this): {ssl_phase_S28*100:5.1f}%')
print(f'\nSup phase val:')
print(f'  nb14 (mer. avg)        : {NB14_PHASE*100:5.1f}%')
print(f'  nb14b Session 25       : {NB14B_S25_PHASE*100:5.1f}%')
print(f'  nb14b Session 28 (this): {sup_phase_S28*100:5.1f}%')

# Three-attempt ablation figure
fig, (axL, axR) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Three-attempt Ablation: meridional avg (nb14/nb15) → Session 25 lat-aware (failed) → Session 28 lat-aware',
             fontsize=13, fontweight='bold')

# Left: month confound F (log scale)
labels_L  = ['SSL\n(nb15 → S25 → S28)', 'Sup\n(S28 only)']
vals_L_baseline = [NB15_MONTH_F, np.nan]            # nb15 ssl, nb14 sup (n/a)
vals_L_S25      = [NB15B_S25_MONTH_F, np.nan]       # nb15b S25 ssl, nb14b S25 sup (n/a here)
vals_L_S28      = [f_ssl_S28, f_sup_S28]            # current run
x = np.arange(len(labels_L))
w = 0.27
b0 = axL.bar(x - w, vals_L_baseline, width=w, color='#888888', alpha=0.85, label='meridional avg (nb14/nb15)')
b1 = axL.bar(x,     vals_L_S25,      width=w, color='#d62728', alpha=0.7,  label='Session 25 lat-aware (failed)')
b2 = axL.bar(x + w, vals_L_S28,      width=w, color='#2ca02c', alpha=0.9,  label='Session 28 lat-aware (this run)')
axL.axhline(50,  color='red', lw=1.5, ls='--', alpha=0.7, label='F = 50 (BSISO acceptance threshold)')
axL.axhline(100, color='red', lw=1.0, ls=':',  alpha=0.5, label='F = 100 (S28 falsification floor)')
axL.set_xticks(x); axL.set_xticklabels(labels_L)
axL.set_ylabel('Angle ANOVA F by month (12 groups)')
axL.set_yscale('log')
axL.set_title('Seasonal Confound (lower is better)', fontsize=11)
for bar, v in zip(b0, vals_L_baseline):
    if not np.isnan(v): axL.text(bar.get_x() + bar.get_width()/2, v * 1.1, f'{v:.1f}', ha='center', fontsize=9, color='#444444')
for bar, v in zip(b1, vals_L_S25):
    if not np.isnan(v): axL.text(bar.get_x() + bar.get_width()/2, v * 1.1, f'{v:.1f}', ha='center', fontsize=9, color='#d62728')
for bar, v in zip(b2, vals_L_S28):
    if not np.isnan(v): axL.text(bar.get_x() + bar.get_width()/2, v * 1.1, f'{v:.1f}', ha='center', fontsize=9, color='#2ca02c', fontweight='bold')
axL.legend(fontsize=8, loc='upper right')
axL.grid(alpha=0.3, which='both')

# Right: phase val (linear)
labels_R = ['Sup\n(nb14 → S25 → S28)', 'SSL\n(nb15 → S25 → S28)']
vals_R_baseline = [NB14_PHASE * 100,        NB15_PHASE * 100]
vals_R_S25      = [NB14B_S25_PHASE * 100,   NB15B_S25_PHASE * 100]
vals_R_S28      = [sup_phase_S28 * 100,     ssl_phase_S28 * 100]
x = np.arange(len(labels_R))
b0 = axR.bar(x - w, vals_R_baseline, width=w, color='#888888', alpha=0.85, label='meridional avg (nb14/nb15)')
b1 = axR.bar(x,     vals_R_S25,      width=w, color='#d62728', alpha=0.7,  label='Session 25 lat-aware (failed)')
b2 = axR.bar(x + w, vals_R_S28,      width=w, color='#1f77b4', alpha=0.9,  label='Session 28 lat-aware (this run)')
axR.axhline(12.5, color='red',  lw=1.0, ls=':',  alpha=0.6, label='12.5% random')
axR.axhline(30,   color='gray', lw=1.0, ls='--', alpha=0.6, label='30% target (SSL)')
axR.axhline(60,   color='gray', lw=1.0, ls='--', alpha=0.6, label='60% target (sup)')
axR.set_xticks(x); axR.set_xticklabels(labels_R)
axR.set_ylabel('Phase val accuracy (%)')
axR.set_title('RMM Phase Recovery (higher is better)', fontsize=11)
for bar, v in zip(b0, vals_R_baseline):
    axR.text(bar.get_x() + bar.get_width()/2, v + 1.5, f'{v:.1f}%', ha='center', fontsize=9, color='#444444')
for bar, v in zip(b1, vals_R_S25):
    axR.text(bar.get_x() + bar.get_width()/2, v + 1.5, f'{v:.1f}%', ha='center', fontsize=9, color='#d62728')
for bar, v in zip(b2, vals_R_S28):
    axR.text(bar.get_x() + bar.get_width()/2, v + 1.5, f'{v:.1f}%', ha='center', fontsize=9, color='#1f77b4', fontweight='bold')
axR.set_ylim(0, 80)
axR.legend(fontsize=8, loc='upper right')
axR.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/ablation_three_attempts.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: ablation_three_attempts.png')

# Aliases used by the report cell below
f_ssl_lat16     = f_ssl_S28
f_sup_lat16     = f_sup_S28
sup_phase_lat16 = sup_phase_S28
ssl_phase_lat16 = ssl_phase_S28

## Cell 10 — Comparison Report (Auto-Generated)

In [ ]:
import time as _t

ssl_advantage_vs_sup = z_ssl > z_sup
ssl_advantage_vs_rmm = z_ssl > z_rmm
month_resolved = f_ssl_lat16 < 50
month_improved = f_ssl_lat16 < 200
phase_sup_target_met = sup_phase_lat16 > 0.60
phase_ssl_target_met = ssl_phase_lat16 > 0.30

# nb14 / nb15 / BSISO references for context
NB14_Z = 12.21
NB15_Z = 13.44
NB14_RHO_TAU0 = 0.639
NB15_RHO_TAU0 = 0.100

report_lines = [
    '=' * 75,
    'MJO LAT16 THREE-WAY COMPARISON REPORT',
    f'Date: {_t.strftime("%Y-%m-%d")}',
    '=' * 75,
    '',
    'Representations (all on lat-aware preprocessing from nb13b):',
    f'  rmm  — Wheeler & Hendon RMM index            N = {len(theta_rmm)} days',
    f'  sup  — Supervised 2D, lat-aware (nb14b)      N = {len(theta_sup)} days',
    f'  ssl  — SSL temporal 2D, lat-aware (nb15b)    N = {len(theta_ssl)} days',
    f'  SSL orientation: {"z₂ negated" if SSL_FLIP else "unflipped"}',
    f'  Sup orientation: {"z₂ negated" if SUP_FLIP else "unflipped"}',
    '',
    'LAG CIRCULAR CORRELATION',
    '-' * 75,
]
for name, rho, n, nb in [
    ('rmm↔sup', rho_rmm_sup, n_rmm_sup, null_rmm_sup),
    ('rmm↔ssl', rho_rmm_ssl, n_rmm_ssl, null_rmm_ssl),
    ('sup↔ssl', rho_sup_ssl, n_sup_ssl, null_sup_ssl),
]:
    peak = np.nanargmax(rho)
    report_lines += [
        f'  {name}:  ρ(τ=0)={rho[lags==0][0]:+.3f}   '
        f'peak ρ={rho[peak]:+.3f} at τ={lags[peak]:+d}d   '
        f'95% null |ρ|={nb:.3f}'
    ]

report_lines += [
    f'  (nb14 baseline: ρ_c(rmm,sup;0) = {NB14_RHO_TAU0:.3f})',
    f'  (nb15 baseline: ρ_c(rmm,ssl;0) = {NB15_RHO_TAU0:.3f})',
    '',
    'AUTOCORRELATION (e-folding decorrelation timescale)',
    '-' * 75,
    f'  RMM         : τ_e = {tau_e_rmm}d   '
    f'ρ(τ=1)={rho_auto_rmm[lags==1][0]:.3f}  '
    f'ρ(τ=10)={rho_auto_rmm[lags==10][0]:.3f}  '
    f'ρ(τ=20)={rho_auto_rmm[lags==20][0]:.3f}',
    f'  Sup-lat16   : τ_e = {tau_e_sup}d   '
    f'ρ(τ=1)={rho_auto_sup[lags==1][0]:.3f}  '
    f'ρ(τ=10)={rho_auto_sup[lags==10][0]:.3f}  '
    f'ρ(τ=20)={rho_auto_sup[lags==20][0]:.3f}',
    f'  SSL-lat16   : τ_e = {tau_e_ssl}d   '
    f'ρ(τ=1)={rho_auto_ssl[lags==1][0]:.3f}  '
    f'ρ(τ=10)={rho_auto_ssl[lags==10][0]:.3f}  '
    f'ρ(τ=20)={rho_auto_ssl[lags==20][0]:.3f}',
    '',
    'ENSO DISPLACEMENT Z-SCORES (1000 permutations, active MJO only)',
    '-' * 75,
    f'  RMM         : z = {z_rmm:+.2f}   (observed {mu_rmm:.4f}, null {b_rmm:.4f}±{s_rmm:.4f})',
    f'  Sup-lat16   : z = {z_sup:+.2f}   (observed {mu_sup:.4f}, null {b_sup:.4f}±{s_sup:.4f})   [nb14 was {NB14_Z:.2f}]',
    f'  SSL-lat16   : z = {z_ssl:+.2f}   (observed {mu_ssl:.4f}, null {b_ssl:.4f}±{s_ssl:.4f})   [nb15 was {NB15_Z:.2f}]',
    '',
    "EN − LN OLR' COMPOSITE max |Δ| (σ units, peak ENSO modulation amplitude, lat-averaged)",
    '-' * 75,
    f'  RMM         : {np.nanmax(np.abs(diff_rmm)):.3f}σ',
    f'  Sup-lat16   : {np.nanmax(np.abs(diff_sup)):.3f}σ',
    f'  SSL-lat16   : {np.nanmax(np.abs(diff_ssl)):.3f}σ',
    f'  (Strongest SSL phase by lat-resolved |Δ|: P{ph_star})',
    '',
    'SESSION 25 ABLATION (lat-aware vs meridional avg)',
    '-' * 75,
    f'  SSL angle ANOVA F :  nb15  = {NB15_MONTH_F:.2f}  →  nb15b = {f_ssl_lat16:.2f}   '
    f'(target < 50; {"PASSED" if month_resolved else "FAILED" if not month_improved else "PARTIAL"})',
    f'  Sup angle ANOVA F :  nb14  =  n/a   →  nb14b = {f_sup_lat16:.2f}   (informational only)',
    f'  SSL phase val     :  nb15  = {NB15_PHASE*100:.1f}% →  nb15b = {ssl_phase_lat16*100:.1f}%   '
    f'(target > 30%; {"PASSED" if phase_ssl_target_met else "FAILED"})',
    f'  Sup phase val     :  nb14  = {NB14_PHASE*100:.1f}% →  nb14b = {sup_phase_lat16*100:.1f}%   '
    f'(target > 60%; {"PASSED" if phase_sup_target_met else "FAILED"})',
    '',
    'BSISO REFERENCE',
    '-' * 75,
    '  64D supervised:           phase 67.7%, z 3.83',
    '  2D supervised (nb07c):    phase 58.3%, z 2.53',
    '  2D SSL temporal (nb08):   phase ~30-40%, z 14.55  ← SSL advantage on BSISO',
    '',
    'HEADLINE COMPARISON',
    '-' * 75,
    f'  SSL advantage over Supervised:   z_SSL ({z_ssl:.2f}) {"  > " if ssl_advantage_vs_sup else " <= "}  z_sup ({z_sup:.2f})',
    f'  SSL advantage over RMM:          z_SSL ({z_ssl:.2f}) {"  > " if ssl_advantage_vs_rmm else " <= "}  z_rmm ({z_rmm:.2f})',
    '',
]

# Layered interpretation
if not month_improved:
    interp = ['INTERPRETATION:',
              f'  Seasonal confound NOT resolved by lat-aware redesign (month F={f_ssl_lat16:.1f}).',
              f'  Treat z_SSL={z_ssl:.2f} as inflated by seasonal contamination, as in nb15.',
              '  Next steps: tighten bandpass to (20, 60) d, or restrict pairs to same calendar month.']
elif month_resolved and ssl_advantage_vs_sup and z_ssl > 5:
    interp = ['INTERPRETATION:',
              '  Lat-aware redesign resolved the seasonal confound (F < 50) AND',
              f'  SSL ENSO advantage survives (z_SSL={z_ssl:.2f} > z_sup={z_sup:.2f} > 5).',
              '  This is the publishable MJO result: the SSL advantage observed in BSISO',
              '  (z_SSL=14.55 vs z_sup=2.53) generalizes to MJO once the meridional axis is preserved.',
              '  Strong evidence that temporal contrastive learning captures genuine intraseasonal',
              '  ENSO-MJO coupling that supervised methods miss.']
elif month_resolved and z_ssl < 5:
    interp = ['INTERPRETATION:',
              f'  Lat-aware redesign resolved the seasonal confound (F={f_ssl_lat16:.1f}) but',
              f'  SSL z dropped to {z_ssl:.2f}. This confirms that the bulk of nb15\'s z=13.44',
              '  was seasonal contamination. The true SSL ENSO signal is closer to the supervised baseline.',
              '  Still a publishable scientific finding — just a different framing than originally hoped.']
elif month_improved and z_ssl > 5:
    interp = ['INTERPRETATION:',
              f'  Partial improvement (month F: 300.84 → {f_ssl_lat16:.1f}; z={z_ssl:.2f}).',
              '  Lat-aware redesign helped but didn\'t fully resolve the seasonal confound.',
              '  Consider tighter bandpass or asymmetric SSL channel widths.']
else:
    interp = ['INTERPRETATION:',
              f'  Mixed signal — month F = {f_ssl_lat16:.1f}, z_SSL = {z_ssl:.2f}.',
              '  Document carefully before drawing conclusions.']

report_lines += interp

report_lines += [
    '',
    'OUTPUT FILES',
    '-' * 75,
    '  lag_circular_corr.png                       — 3-panel pairwise lag correlation',
    '  lag_circular_corr_overlay.png               — overlay of all three cross-correlation curves',
    '  autocorrelation.png                         — e-folding timescales + autocorrelation overlay',
    '  enso_displacement_3way.png                  — per-phase ENSO displacement bars',
    "  phase_composites_olr.png                    — OLR' longitude profiles (lat-averaged)",
    '  enln_difference_composites.png              — EN−LN difference 1D heatmaps (lat-averaged)',
    '  enln_lat_resolved_strongest_phase.png       — NEW: lat-resolved EN-LN map at strongest SSL phase',
    '  ablation_lat16_vs_meridional_avg.png        — NEW: Session 25 ablation (month F + phase val)',
    '  mjo_comparison_lat16_report.txt             — this report',
    '  mjo_comparison_lat16_summary.csv            — numerical summary',
    '=' * 75,
]

report_text = '\n'.join(report_lines)
print(report_text)

with open(f'{OUT_DIR}/mjo_comparison_lat16_report.txt', 'w') as f:
    f.write(report_text)
print(f'\nSaved: {OUT_DIR}/mjo_comparison_lat16_report.txt')

# Numerical summary CSV (extends nb16's with month F + ablation deltas)
df_summary = pd.DataFrame([
    {'rep': 'rmm', 'rho_tau0_with_rmm': 1.0,
     'z_score': z_rmm, 'max_enln_olr_sigma': float(np.nanmax(np.abs(diff_rmm))),
     'tau_e_days': tau_e_rmm, 'month_F_angle': np.nan,
     'phase_val': np.nan},
    {'rep': 'sup_lat16', 'rho_tau0_with_rmm': float(rho_rmm_sup[lags==0][0]),
     'z_score': z_sup, 'max_enln_olr_sigma': float(np.nanmax(np.abs(diff_sup))),
     'tau_e_days': tau_e_sup, 'month_F_angle': float(f_sup_lat16),
     'phase_val': sup_phase_lat16},
    {'rep': 'ssl_lat16', 'rho_tau0_with_rmm': float(rho_rmm_ssl[lags==0][0]),
     'z_score': z_ssl, 'max_enln_olr_sigma': float(np.nanmax(np.abs(diff_ssl))),
     'tau_e_days': tau_e_ssl, 'month_F_angle': float(f_ssl_lat16),
     'phase_val': ssl_phase_lat16},
])
df_summary.to_csv(f'{OUT_DIR}/mjo_comparison_lat16_summary.csv', index=False)
print(f'Saved: {OUT_DIR}/mjo_comparison_lat16_summary.csv')
print(df_summary.to_string(index=False))

## Cell 11 — (Optional) Download All Outputs

In [ ]:
from google.colab import files
for fname in sorted(os.listdir(OUT_DIR)):
    files.download(f'{OUT_DIR}/{fname}')

---
## Done!

MJO lat-aware pipeline is now functionally complete. Files produced:

```
BSISO_SSL_Project/MJO/lat16/results/comparison/
├── lag_circular_corr.png                       ← 3-panel pairwise lag correlation
├── lag_circular_corr_overlay.png               ← overlay of all three cross-correlation curves
├── autocorrelation.png                         ← e-folding timescales + autocorrelation overlay
├── enso_displacement_3way.png                  ← per-phase ENSO displacement bars
├── phase_composites_olr.png                    ← phase composite OLR' longitude profiles (lat-averaged)
├── enln_difference_composites.png              ← EN-LN difference 1D heatmaps (lat-averaged)
├── enln_lat_resolved_strongest_phase.png       ← NEW: lat-resolved EN-LN map at strongest SSL phase
├── ablation_lat16_vs_meridional_avg.png        ← NEW: Session 25 ablation panels
├── mjo_comparison_lat16_report.txt             ← auto-generated comparison report
└── mjo_comparison_lat16_summary.csv            ← numerical summary (incl. month F, phase val)
```

**Send back:**
1. `mjo_comparison_lat16_report.txt` — headline numbers + interpretation
2. `ablation_lat16_vs_meridional_avg.png` — **the Session 25 verdict** (did the redesign work?)
3. `enln_lat_resolved_strongest_phase.png` — what does the lat axis show us that nb16 couldn't?
4. `enso_displacement_3way.png` — per-phase z comparison
5. `lag_circular_corr.png` — how the three representations relate
6. `autocorrelation.png` — temporal memory across reps

---
*DDCS Project | jh9141@nyu.edu*